# Mava on Colab — Real-Budget Experiments

This notebook sets up the Mava MARL framework on a Colab Pro runtime (A100 recommended) and runs the redesigned experiment study with proper training budgets.

## Research questions (kept from the original plan)

1. How do `ff_ippo` and `ff_mappo` behave on cooperative warehouse-style tasks when scenario difficulty, seed, and recurrence are controlled?
2. Which algorithm performs best under increasing task difficulty?
3. Which algorithm is most stable across random seeds?
4. How sensitive are the algorithms to hyperparameter changes (`actor_lr`)?
5. Does recurrent memory help in partially observable settings?
6. Which method is more sample-efficient under the same training budget?

## What is different vs. the previous (CPU) study

The earlier `analysis-report` branch ran every experiment with `num_envs=1`, `rollout_length=16`, `num_updates=50` → only **800 environment steps** per run. That budget is roughly 10⁴× below the RWARE PPO training budget used in the literature, so no method had a chance to learn. This notebook re-runs the study with budgets that can actually produce learning signal.

---
## 1. Verify GPU runtime

Before running the cells below, make sure Colab is on a GPU runtime:

**Runtime → Change runtime type → Hardware accelerator → GPU → GPU type: A100 (Colab Pro)**

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


---
## 2. Mount Google Drive (for persistent logs)

Colab sessions are ephemeral. We save all run outputs to Drive so they survive disconnects.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

RUN_ROOT = '/content/drive/MyDrive/mava_colab_runs'
!mkdir -p {RUN_ROOT}
print('Run outputs will be saved under:', RUN_ROOT)

Mounted at /content/drive
Run outputs will be saved under: /content/drive/MyDrive/mava_colab_runs


---
## 3. Clone the repository

Clones the `colab-real-experiments` branch from the `hasanbarisgok/Mava` fork. Make sure the branch has been pushed to GitHub before running this cell.

In [ ]:
REPO_URL = 'https://github.com/hasanbarisgok/Mava.git'
BRANCH   = 'colab-real-experiments'

%cd /content
!rm -rf Mava
!git clone -b {BRANCH} {REPO_URL} Mava
%cd /content/Mava
!git rev-parse --short HEAD

---
## 4. Install JAX (CUDA 12) and Mava

The repo pins `jax==0.5.3`. We install the matching CUDA 12 wheels first, then install Mava in editable mode. Git-based dependencies (`jaxmarl`, `marl-eval`, `smaclite`, `gigastep`) will be pulled automatically.

Install can take 5–10 minutes on first run.

In [ ]:
!pip install -q --upgrade pip
!pip install -q "jax[cuda12]==0.5.3" "jaxlib==0.5.3"
!pip install -q -e ".[cuda12]"

In [ ]:
import jax
print('JAX version :', jax.__version__)
print('Devices     :', jax.devices())
print('Default bk  :', jax.default_backend())
assert jax.default_backend() == 'gpu', 'JAX is not on GPU. Re-check runtime type and CUDA install.'

---
## 5. Smoke test

A short run to confirm the pipeline executes end-to-end on GPU. This is **not** a real experiment — it just verifies the install.

In [ ]:
!python mava/systems/ppo/anakin/ff_ippo.py \
  env=rware env/scenario=tiny-2ag \
  system.seed=1 \
  system.num_updates=10 \
  arch.num_envs=4 \
  system.rollout_length=32 \
  system.num_minibatches=2 \
  system.update_batch_size=1 \
  arch.num_evaluation=2 \
  arch.num_eval_episodes=4 \
  logger.loggers.tensorboard.enabled=False \
  logger.loggers.json.enabled=False

---
## 6. Real-budget protocol

These are the parameters proposed for the main study. They follow Mava defaults more closely than the previous run and use vectorized envs on the A100.

| Parameter | Value | Note |
| --- | ---: | --- |
| `arch.num_envs` | `64` | parallel envs vectorised by JAX |
| `system.rollout_length` | `128` | matches Mava default |
| `system.num_minibatches` | `2` | matches Mava default |
| `system.update_batch_size` | `2` | matches Mava default |
| `system.num_updates` | `500` | gives ~16M env steps per run |
| `arch.num_evaluation` | `10` | richer learning curve |
| `arch.num_eval_episodes` | `32` | lower variance per eval |
| `arch.absolute_metric` | `True` | use absolute final-metric eval |

Total timesteps per run ≈ `num_updates × update_batch_size × num_envs × rollout_length` = `500 × 2 × 64 × 128 ≈ 8.2M`. On A100 each run is expected to be in the single-digit minutes.

The cell below runs **one** full-budget seed to confirm timing before scaling up to the full matrix.

In [ ]:
RUN_TAG = 'real_v1_pilot'
EXP_PATH = f'{RUN_ROOT}/{RUN_TAG}'
!mkdir -p {EXP_PATH}

!python mava/systems/ppo/anakin/ff_ippo.py \
  env=rware env/scenario=tiny-2ag \
  system.seed=1 \
  system.num_updates=500 \
  arch.num_envs=64 \
  system.rollout_length=128 \
  system.num_minibatches=2 \
  system.update_batch_size=2 \
  arch.num_evaluation=10 \
  arch.num_eval_episodes=32 \
  arch.num_absolute_metric_eval_episodes=32 \
  arch.absolute_metric=True \
  logger.base_exp_path={EXP_PATH} \
  logger.loggers.tensorboard.enabled=True \
  logger.loggers.json.enabled=True

---
## 7. Batch runner (next step)

Once the pilot run above looks healthy (non-zero late-stage episode return, sensible runtime), the next step is to expand to the full experiment matrix:

- algorithms: `ff_ippo`, `ff_mappo`, `rec_ippo`, `rec_mappo`
- scenarios: `tiny-2ag`, `tiny-4ag`, `tiny-4ag-easy`, `small-4ag`
- seeds: `1..5`

That's 4 × 4 × 5 = 80 main-study runs. With ~3–5 min per run on A100 the full matrix fits in a single Colab Pro session.

A batch driver script will be added in a follow-up commit (e.g. `experiments/run_matrix.py`) once the pilot run is validated.

---
## 8. Monitoring (optional)

TensorBoard inside Colab:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_ROOT}